# Huấn luyện mô hình khử nhiễu ECG (Phase 1) trên Google Colab
Notebook này được thiết lập tự động để sao chép mã nguồn, tải dữ liệu cần thiết từ PhysioNet, và lưu checkpoint trực tiếp lên Google Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Tải mã nguồn và cài đặt thư viện

In [ ]:
%cd /content
import getpass

token = getpass.getpass('Nhập GitHub Personal Access Token (PAT): ')

# Dùng strip() để dọn dẹp khoảng trắng hoặc ký tự xuống dòng thừa nếu lỡ copy dính
repo_url = f"https://{token.strip()}@github.com/vzyhug/phase1.git"

# Tiến hành clone với URL đã được làm sạch
!git clone {repo_url}

%cd phase1
!pip install -r requirements.txt

/content
Nhập GitHub Personal Access Token (PAT): ··········
Cloning into 'phase1'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 48 (delta 10), reused 43 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 16.61 KiB | 1.84 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/phase1


## 2. Chuẩn bị dữ liệu (QT Database & MIT-BIH NST)
Tải trực tiếp từ PhysioNet và giải nén vào thư mục `data/raw`.

In [ ]:
import os

# Tạo các thư mục dữ liệu
os.makedirs('data/raw/qt_database', exist_ok=True)
os.makedirs('data/raw/mit_bih_nst', exist_ok=True)

# Tải và giải nén QT Database
!wget -q -O qt.zip https://physionet.org/static/published-projects/qtdb/qt-database-1.0.0.zip
!unzip -q -j qt.zip -d data/raw/qt_database

# Tải và giải nén MIT-BIH Noise Stress Test Database
!wget -q -O nst.zip https://physionet.org/static/published-projects/nstdb/mit-bih-noise-stress-test-database-1.0.0.zip
!unzip -q -j nst.zip -d data/raw/mit_bih_nst

!rm qt.zip nst.zip
print("Tải và giải nén dữ liệu hoàn tất.")

replace data/raw/qt_database/index.shtml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Tải và giải nén dữ liệu hoàn tất.


## 3. Kết nối thư mục Checkpoints với Google Drive
Bằng cách này, các tệp model checkpoint (như `model.pth`) sẽ được tự động đồng bộ và lưu vào Google Drive, bạn sẽ không bị mất model khi phiên Colab ngắt kết nối.

In [ ]:
import os
drive_ckpt_path = '/content/drive/MyDrive/Phase1_Checkpoints'
os.makedirs(drive_ckpt_path, exist_ok=True)

# Xoá thư mục checkpoints mặc định nếu có và tạo symbolic link tới Drive
!rm -rf checkpoints
!ln -s {drive_ckpt_path} checkpoints
print(f"Thư mục checkpoints đã được liên kết tới {drive_ckpt_path}")

Thư mục checkpoints đã được liên kết tới /content/drive/MyDrive/Phase1_Checkpoints


## 4. Tiền xử lý dữ liệu (Preprocessing)
Quá trình này sẽ thực hiện resampling, phân đoạn (segment), chuẩn hóa và tổng hợp dữ liệu nhiễu (synthesize) để tạo file `.npy`.

In [ ]:
!python run_workflow.py --mode preprocess

=== Preprocessing: Generating synthesized data ===
Preprocessing data from scratch...
Resampling completed. Output: data/processed/clean_360hz
Traceback (most recent call last):
  File "/content/phase1/run_workflow.py", line 46, in <module>
    main(args.mode)
  File "/content/phase1/run_workflow.py", line 20, in main
    Data_Preparation(n_type=1, force_rebuild=True, test_records=TEST_RECORDS)
  File "/content/phase1/src/data_prep/data_preparation.py", line 35, in Data_Preparation
    synthesize_noisy(seg_norm, 'data/raw/mit_bih_nst', data_dir)
  File "/content/phase1/src/data_prep/synthesizer.py", line 9, in synthesize_noisy
    noise_sig, _ = wfdb.rdsamp(noise_path)
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wfdb/io/record.py", line 2346, in rdsamp
    record = rdrecord(
             ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wfdb/io/record.py", line 2052, in rdrecord
    record = rdheader(record_name, pn_dir=pn_dir, rd_

## 5. Huấn luyện (Training)
Bắt đầu huấn luyện mô hình. Bạn có thể điều chỉnh số epoch, batch size hoặc learning rate trong tệp `configs/base.yaml`.

In [ ]:
!python run_workflow.py --mode train

=== Training model ===
Preprocessing data from scratch...
Resampling completed. Output: data/processed/clean_360hz
Traceback (most recent call last):
  File "/content/phase1/run_workflow.py", line 46, in <module>
    main(args.mode)
  File "/content/phase1/run_workflow.py", line 25, in main
    train_model('configs/base.yaml', device='cuda:0' if torch.cuda.is_available() else 'cpu')
  File "/content/phase1/src/training/trainer.py", line 17, in train_model
    X_train, y_train, X_test, y_test = Data_Preparation(n_type=1, force_rebuild=False)
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/phase1/src/data_prep/data_preparation.py", line 35, in Data_Preparation
    synthesize_noisy(seg_norm, 'data/raw/mit_bih_nst', data_dir)
  File "/content/phase1/src/data_prep/synthesizer.py", line 9, in synthesize_noisy
    noise_sig, _ = wfdb.rdsamp(noise_path)
                   ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-pack

## 6. Kiểm tra Inference (Tùy chọn)

In [ ]:
!python run_workflow.py --mode infer

=== Running inference on a sample ===
Traceback (most recent call last):
  File "/content/phase1/run_workflow.py", line 46, in <module>
    main(args.mode)
  File "/content/phase1/run_workflow.py", line 30, in main
    noisy_sample = np.load('data/synthesis/noisy.npy')[0]   # (512,)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 455, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/synthesis/noisy.npy'
